# Train with KubeRay, Kueue, and autoscaling

This notebook shows how to submit distributed training jobs with the **Alauda Build of KubeRay Operator** and the **Alauda Build of Kueue**. It covers two scheduling modes:

1. Kueue admits a fixed-size `RayJob` as an all-or-nothing group.
2. Kueue admits an elastic `RayJob`, then Ray Autoscaler v2 requests additional workers as the training workload grows.

Kueue controls *when* a workload may consume quota. Ray Autoscaler controls the number of Ray worker Pods inside that admitted workload. A Kubernetes Cluster Autoscaler or node provisioner is still required when new Pods need new nodes.

```text
RayJob -> Kueue Workload -> RayCluster (head + workers)
                              ^             ^
                    Kueue admits quota   Ray Autoscaler changes workers
```

The examples use the `ray.io/v1` API, KubeRay Operator 1.6.x, Kueue 0.17.x, and a Ray image that includes the training framework. In this notebook, a training job means a KubeRay `RayJob`; it is distinct from a Kubeflow Trainer `TrainJob`. Keep the Ray version in the image and in `rayClusterSpec.rayVersion` aligned. The reusable manifests are in `assets/kuberay-training-with-kueue-autoscaling/`; keep that directory beside this notebook. From a Workbench terminal, download both paths with a sparse checkout:

```bash
git clone --depth 1 --filter=blob:none --sparse https://github.com/alauda/aml-docs.git
git -C aml-docs sparse-checkout set --no-cone \
  /docs/en/train/guides/kuberay-training-with-kueue-autoscaling.ipynb \
  /docs/en/train/guides/assets/kuberay-training-with-kueue-autoscaling/
cd aml-docs/docs/en/train/guides
```

## Prerequisites

- The Alauda Build of KubeRay Operator is installed and `rayclusters.ray.io` and `rayjobs.ray.io` are available.
- The Alauda Build of Kueue is installed and configured to manage `ray.io/rayjob` and `ray.io/raycluster`.
- A namespace is available for training jobs. The namespace must have a Kueue `LocalQueue`.
- An approved image is available in the cluster registry. It must contain Ray and the training code or dependencies.
- The image pull Secret, node selectors, accelerator resources, and security context are configured for the target cluster.
- For elastic jobs, the Kueue feature gate `ElasticJobsViaWorkloadSlices` is enabled by a cluster administrator.

Kueue-managed `RayJob` resources must create their own RayCluster through `spec.rayClusterSpec`. They cannot use `clusterSelector` to attach to an existing RayCluster. Set `shutdownAfterJobFinishes: true` so the job-owned RayCluster is released after the run.

In [ ]:
%%bash
set -euo pipefail
NAMESPACE="${KUBERAY_KUEUE_NAMESPACE:-ray-training}"
RAY_IMAGE="${KUBERAY_TRAINING_IMAGE:-}"
: "${RAY_IMAGE:?Set KUBERAY_TRAINING_IMAGE to an approved Ray training image}"
kubectl get crd rayclusters.ray.io
kubectl get crd rayjobs.ray.io
kubectl get deploy -n cpaas-system kuberay-operator kueue-controller-manager \
  -o custom-columns=NAME:.metadata.name,IMAGE:.spec.template.spec.containers[0].image
kubectl get namespace "$NAMESPACE" --show-labels
kubectl get localqueue -n "$NAMESPACE"


## Configure Kueue quota

A cluster administrator creates a `ResourceFlavor` and `ClusterQueue`. A namespace administrator creates a `LocalQueue` that points to the `ClusterQueue`. Users reference only the `LocalQueue` in the RayJob label.

The example below is for homogeneous CPU and memory nodes. For GPUs, define a ResourceFlavor with the appropriate node labels and cover the device resource used by the device plugin, such as `nvidia.com/gpu` or an Alauda HAMI resource. Set quotas from the real capacity and tenant policy.

In [ ]:
%%bash
set -euo pipefail
NAMESPACE="${KUBERAY_KUEUE_NAMESPACE:-ray-training}"
CLUSTER_QUEUE="${KUBERAY_CLUSTER_QUEUE:-ray-cq}"
LOCAL_QUEUE="${KUBERAY_LOCAL_QUEUE:-ray-lq}"
ASSET_DIR="${KUBERAY_KUEUE_ASSET_DIR:-assets/kuberay-training-with-kueue-autoscaling}"
kubectl create namespace "$NAMESPACE" --dry-run=client -o yaml | kubectl apply -f -
sed "s#ray-cq#$CLUSTER_QUEUE#g" "$ASSET_DIR/kueue-resources.yaml" | kubectl apply -f -
sed -e "s#ray-lq#$LOCAL_QUEUE#g" -e "s#<namespace>#$NAMESPACE#g" \
  "$ASSET_DIR/local-queue.yaml" | kubectl apply -f -
kubectl get clusterqueue "$CLUSTER_QUEUE"
kubectl get localqueue "$LOCAL_QUEUE" -n "$NAMESPACE"

## Enable elastic workloads for Kueue

Kueue's elastic Ray integration uses WorkloadSlices. The cluster administrator must enable the `ElasticJobsViaWorkloadSlices` feature gate in the Kueue `Configuration` before submitting the autoscaling example. The exact installation mechanism is platform-specific; configure it through the Alauda Build of Kueue deployment values or upgrade form.

The relevant section of the Kueue controller configuration is:

```yaml
apiVersion: config.kueue.x-k8s.io/v1beta2
kind: Configuration
featureGates:
  ElasticJobsViaWorkloadSlices: true
```

On the dev cluster, the controller configuration is mounted from `cpaas-system/kueue-manager-config`. Do not patch a Helm-managed ConfigMap as a production change; update the Kueue installation configuration and let it roll the controller. Verify the effective setting before using the elastic cells.

In [ ]:
%%bash
set -euo pipefail
kubectl -n cpaas-system get configmap kueue-manager-config \
  -o jsonpath='{.data.controller_manager_config\.yaml}' | \
  grep -n -E 'featureGates|ElasticJobsViaWorkloadSlices|ray.io/ray(job|cluster)' || true
echo
echo "If ElasticJobsViaWorkloadSlices is absent, ask the Kueue administrator to enable it before continuing."

## Submit a fixed-size training RayJob

This job requests one head and one worker. The queue label is the Kueue admission boundary. Kueue keeps `spec.suspend` true until the workload is admitted, then unsuspends the RayJob and KubeRay creates the RayCluster. The entrypoint below is a small distributed-looking workload; replace it with the training command in your image.

For real training, request the complete head and worker CPU/memory/GPU footprint in `rayClusterSpec`. Kueue admits the complete set together, so under-requesting resources can cause a job to start and then fail when Ray tries to place its tasks.

In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${KUBERAY_KUEUE_ASSET_DIR:-assets/kuberay-training-with-kueue-autoscaling}"
NAMESPACE="${KUBERAY_KUEUE_NAMESPACE:-ray-training}"
LOCAL_QUEUE="${KUBERAY_LOCAL_QUEUE:-ray-lq}"
RAY_IMAGE="${KUBERAY_TRAINING_IMAGE:?Set KUBERAY_TRAINING_IMAGE}"
JOB_NAME="${KUBERAY_FIXED_JOB_NAME:-ray-train-$(date +%Y%m%d-%H%M%S)}"
sed -e "s#<job-name>#$JOB_NAME#g" -e "s#<queue-name>#$LOCAL_QUEUE#g" \
  -e "s#<image>#$RAY_IMAGE#g" "$ASSET_DIR/rayjob-fixed.yaml" | \
  kubectl apply -n "$NAMESPACE" -f -
echo "Submitted $JOB_NAME"
echo "For the monitor and cleanup cells, set KUBERAY_FIXED_JOB_NAME=$JOB_NAME"
kubectl -n "$NAMESPACE" get rayjob "$JOB_NAME" -o wide
kubectl -n "$NAMESPACE" get workloads -o wide

In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${KUBERAY_KUEUE_ASSET_DIR:-assets/kuberay-training-with-kueue-autoscaling}"
NAMESPACE="${KUBERAY_KUEUE_NAMESPACE:-ray-training}"
JOB_NAME="${KUBERAY_FIXED_JOB_NAME:?Set KUBERAY_FIXED_JOB_NAME to monitor the submitted job}"
for _ in $(seq 1 120); do
  status="$(kubectl -n "$NAMESPACE" get rayjob "$JOB_NAME" -o jsonpath='{.status.jobStatus}' 2>/dev/null || true)"
  echo "RayJob=${status:-PENDING}"
  kubectl -n "$NAMESPACE" get workloads -o wide --no-headers 2>/dev/null || true
  case "$status" in SUCCEEDED) break;; FAILED) kubectl -n "$NAMESPACE" describe rayjob "$JOB_NAME"; exit 1;; esac
  sleep 5
done
[ "${status:-}" = SUCCEEDED ] || { kubectl -n "$NAMESPACE" describe rayjob "$JOB_NAME"; exit 1; }
kubectl -n "$NAMESPACE" get rayjob "$JOB_NAME" -o wide

## Submit an autoscaling training RayJob

An elastic Kueue workload needs both sides of the integration:

- the RayJob has the queue label and `kueue.x-k8s.io/elastic-job: "true"`;
- Kueue has `ElasticJobsViaWorkloadSlices` enabled; and
- the embedded RayCluster enables in-tree autoscaling and gives its worker group a bounded `minReplicas`/`maxReplicas` range.

The workload below starts with one two-CPU worker and creates eight one-CPU Ray tasks that run concurrently. Ray Autoscaler requests more workers, while Kueue admits each scale-up through a WorkloadSlice. After the tasks finish, `idleTimeoutSeconds` allows workers to scale down. This is a wiring test, not a capacity benchmark.

In [ ]:
%%bash
set -euo pipefail
ASSET_DIR="${KUBERAY_KUEUE_ASSET_DIR:-assets/kuberay-training-with-kueue-autoscaling}"
NAMESPACE="${KUBERAY_KUEUE_NAMESPACE:-ray-training}"
LOCAL_QUEUE="${KUBERAY_LOCAL_QUEUE:-ray-lq}"
RAY_IMAGE="${KUBERAY_TRAINING_IMAGE:?Set KUBERAY_TRAINING_IMAGE}"
JOB_NAME="${KUBERAY_ELASTIC_JOB_NAME:-ray-train-elastic-$(date +%Y%m%d-%H%M%S)}"
sed -e "s#<job-name>#$JOB_NAME#g" -e "s#<queue-name>#$LOCAL_QUEUE#g" \
  -e "s#<image>#$RAY_IMAGE#g" "$ASSET_DIR/rayjob-autoscaling.yaml" | \
  kubectl apply -n "$NAMESPACE" -f -
echo "Submitted $JOB_NAME"
echo "For the monitor and cleanup cells, set KUBERAY_ELASTIC_JOB_NAME=$JOB_NAME"
kubectl -n "$NAMESPACE" get rayjob "$JOB_NAME" -o wide
kubectl -n "$NAMESPACE" get workloads -o wide

In [ ]:
%%bash
set -euo pipefail
NAMESPACE="${KUBERAY_KUEUE_NAMESPACE:-ray-training}"
JOB_NAME="${KUBERAY_ELASTIC_JOB_NAME:?Set KUBERAY_ELASTIC_JOB_NAME to monitor the submitted job}"
for _ in $(seq 1 180); do
  status="$(kubectl -n "$NAMESPACE" get rayjob "$JOB_NAME" -o jsonpath='{.status.jobStatus}' 2>/dev/null || true)"
  cluster="$(kubectl -n "$NAMESPACE" get rayjob "$JOB_NAME" -o jsonpath='{.status.rayClusterName}' 2>/dev/null || true)"
  workers="$(kubectl -n "$NAMESPACE" get raycluster "$cluster" -o jsonpath='{.status.availableWorkerReplicas}/{.status.desiredWorkerReplicas}' 2>/dev/null || true)"
  echo "RayJob=${status:-PENDING} RayCluster=${cluster:-PENDING} workers=${workers:-PENDING}"
  kubectl -n "$NAMESPACE" get workloads -o wide --no-headers 2>/dev/null || true
  case "$status" in SUCCEEDED) break;; FAILED) kubectl -n "$NAMESPACE" describe rayjob "$JOB_NAME"; exit 1;; esac
  sleep 5
done
[ "${status:-}" = SUCCEEDED ] || { kubectl -n "$NAMESPACE" describe rayjob "$JOB_NAME"; exit 1; }
kubectl -n "$NAMESPACE" get rayjob "$JOB_NAME" -o wide

## Reusable elastic RayCluster

A long-lived `RayCluster` can also be managed by Kueue and autoscaled. The complete manifest is [`raycluster-autoscaling.yaml`](https://github.com/alauda/aml-docs/tree/master/docs/en/train/guides/assets/kuberay-training-with-kueue-autoscaling/raycluster-autoscaling.yaml). It includes the queue label, elastic annotation, and bounded Ray Autoscaler v2 worker range. Apply it after replacing the image: 

```bash
ASSET_DIR="${KUBERAY_KUEUE_ASSET_DIR:-assets/kuberay-training-with-kueue-autoscaling}"
sed "s#<image>#$KUBERAY_TRAINING_IMAGE#g" "$ASSET_DIR/raycluster-autoscaling.yaml" | kubectl apply -n "$KUBERAY_KUEUE_NAMESPACE" -f -
```

Unlike the Kueue-managed RayJob example, this cluster holds quota for its lifetime, so delete it when the workload is idle. Kueue's Ray integration has an important lifecycle constraint: a Kueue-managed RayJob cannot select this existing cluster. Use a queue-managed RayJob with an embedded `rayClusterSpec` when each training run needs independent admission and cleanup.

## Monitor admission, scaling, and failures

Use Kubernetes status for queue admission and KubeRay status for cluster scaling:

```bash
kubectl -n $NAMESPACE get rayjob,raycluster,pods -o wide
kubectl -n $NAMESPACE get workloads -o wide
kubectl -n $NAMESPACE describe workload <workload-name>
kubectl -n $NAMESPACE describe rayjob <job-name>
kubectl -n $NAMESPACE describe raycluster <cluster-name>
kubectl -n $NAMESPACE logs <head-pod> -c autoscaler
kubectl -n cpaas-system logs deploy/kueue-controller-manager
```

Check for an `Admitted` Workload condition, `spec.suspend` changing to `false`, WorkloadSlices during scale-up, and worker replica changes. If the workload is admitted but workers remain pending, inspect node capacity, image-pull events, ResourceFlavor node labels, and the ClusterQueue quota. Ray Autoscaler cannot create Kubernetes nodes.

For production training, checkpoint to durable object storage or RWX storage before enabling preemption or elastic scaling. Record the job name, image digest, source revision, dataset version, worker bounds, queue, and final Ray/Kueue statuses as run evidence.

## Production checklist

- Keep `minReplicas` and `maxReplicas` within a tested quota and node-capacity envelope.
- Use separate ResourceFlavors and ClusterQueues for CPU, GPU, and other accelerator pools.
- Set a priority policy and preemption behavior deliberately; do not allow low-priority elastic jobs to consume an unbounded borrowing quota.
- Keep the number of Ray worker groups within Kueue's Workload PodSet limit.
- Pin images by digest and keep Ray, KubeRay, Kueue, and training framework versions compatible.
- Use `shutdownAfterJobFinishes: true` and a retention policy for queue-managed RayJobs.
- Configure checkpointing and idempotent output paths so retries or preemption do not lose progress or create duplicate artifacts.
- Validate autoscaling with production-shaped task parallelism, skew, memory pressure, and image-pull latency before setting an SLA.

In [ ]:
%%bash
set -euo pipefail
NAMESPACE="${KUBERAY_KUEUE_NAMESPACE:-ray-training}"
kubectl -n "$NAMESPACE" delete rayjob "${KUBERAY_FIXED_JOB_NAME:-ray-train-placeholder}" --ignore-not-found
kubectl -n "$NAMESPACE" delete rayjob "${KUBERAY_ELASTIC_JOB_NAME:-ray-train-elastic-placeholder}" --ignore-not-found
echo "Delete any remaining RayClusters only after confirming that no other job uses them."

## References

- [Alauda Build of KubeRay Operator](../../develop/components/kuberay/intro.mdx)
- [Install KubeRay Operator](../../develop/components/kuberay/install.mdx)
- [Alauda Build of Kueue](../components/kueue/intro.mdx)
- [KubeRay and Kueue integration](https://docs.ray.io/en/latest/cluster/kubernetes/k8s-ecosystem/kueue.html)
- [KubeRay autoscaling](https://docs.ray.io/en/latest/cluster/kubernetes/user-guides/configuring-autoscaling.html)
- [Run a RayJob with Kueue](https://kueue.sigs.k8s.io/docs/tasks/run/rayjobs/)
- [Kueue elastic workloads](https://kueue.sigs.k8s.io/docs/concepts/elastic_workload/)